
# IMU-SMPL Mesh Matching Demo

`Demo.ipynb`의 구도를 유지하되, `IMU ↔ SMPL` 매칭으로 바꾼 데모입니다.

- 상단: 여러 사람 SMPL **mesh** 렌더
- 하단 좌: Query IMU signal
- 하단 우: Query IMU와 각 SMPL 후보의 유사도
- 매 프레임: 예측된 매칭 SMPL만 강조색으로 표시

기본 설정은 `gt_pose` + `wandb/latest-run/files/model_*.pth` 입니다.


In [53]:

import os
import re
import glob
import copy
import random
import pickle
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from tqdm import tqdm
import imageio
import inspect
if not hasattr(inspect, "getargspec"):
    inspect.getargspec = inspect.getfullargspec
import smplx

import sys

# Compatibility for legacy SMPL/chumpy pickles
for name, val in {
    "bool": bool,
    "int": int,
    "float": float,
    "complex": complex,
    "object": object,
    "str": str,
    "unicode": str,
}.items():
    if not hasattr(np, name):
        setattr(np, name, val)

import ctypes
# Fix libstdc++ ABI mismatch for pytorch3d extension on this host
ctypes.CDLL("/usr/lib/x86_64-linux-gnu/libstdc++.so.6", mode=ctypes.RTLD_GLOBAL)

try:
    from mmhuman3d.core.visualization.visualize_smpl import (
        visualize_smpl_calibration,
        visualize_smpl_pose,
    )
    from mmhuman3d.models.body_models.builder import build_body_model
    MMHUMAN3D_AVAILABLE = True
except Exception as _mmhuman3d_err:
    visualize_smpl_calibration = None
    visualize_smpl_pose = None
    build_body_model = None
    MMHUMAN3D_AVAILABLE = False

from src.evaluation import matching
from src.models import model_loader, SPITE



In [54]:

# -------------------------
# Config
# -------------------------
SEED = 1337
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

GPU_INDEX = 1
RENDER_RESOLUTION = (512, 1024)

if torch.cuda.is_available():
    torch.cuda.set_device(GPU_INDEX)
    device = f"cuda:{GPU_INDEX}"
else:
    device = "cpu"
print("Device:", device)

# Matching/data config
DATASET_SPLIT = "eLIPD"
MODEL_TYPE = "SIE"
SKELETON_SOURCE = "gt_pose"   # 핵심: gt_joint가 아니라 gt_pose
EMBED_DIM = 512
NUM_JOINTS = 24
N_FEATS = 6 if SKELETON_SOURCE == "gt_pose" else 3

# Scene config
N_SUBJECTS = 8
WINDOW_SIZE = 160
SCENE_SEED = 42
FPS = 15
MESH_SCALE = 30
SUBJECT_SPACING = 20  # 사람 간격 (겹치면 이 값 더 키우기)
MMHUMAN_SUBJECT_SPACING = 0.9  # mmhuman3d 카메라용 간격
X_MARGIN = 1.5

# Mesh render config
MESH_FACE_STRIDE = 8   # 작을수록 디테일 높고 느림
MESH_ALPHA_OTHER = 0.65
MESH_ALPHA_MATCH = 0.95
MESH_COLOR_OTHER = "#bdbdbd"
MESH_COLOR_MATCH = "#d7191c"
MESH_COLOR_QUERY = "#2c7bb6"

# Paths
DATA_PATH_CANDIDATES = [
    "data/LIPD/LIPD_SEQUENCES_256p.pkl",
    "/data/LIPD/LIPD_SEQUENCES_256p.pkl",
]

SMPL_MODEL_CANDIDATES = [
    "/home/kdh/despite/smpl/SMPL_NEUTRAL.pkl",
]

OUTPUT_PREFIX = "imu_smpl_mesh_matching"


Device: cuda:1


In [55]:

def resolve_first_existing(paths):
    for p in paths:
        if os.path.exists(p):
            return p
    raise FileNotFoundError("No existing path among: {}".format(paths))


def resolve_latest_wandb_model(latest_run_dir="wandb/latest-run/files"):
    pattern = os.path.join(latest_run_dir, "model_*.pth")
    ckpts = glob.glob(pattern)
    if not ckpts:
        raise FileNotFoundError("No model_*.pth found under {}".format(latest_run_dir))

    def epoch_of(path):
        m = re.search(r"model_(\d+)\.pth$", os.path.basename(path))
        return int(m.group(1)) if m else -1

    ckpts = sorted(ckpts, key=epoch_of)
    return ckpts[-1]




def resolve_smpl_model_dir(model_pkl_path):
    # mmhuman3d build_body_model expects root dir that contains `smpl/`.
    model_dir = os.path.dirname(model_pkl_path)
    if os.path.basename(model_dir).lower() == "smpl":
        model_dir = os.path.dirname(model_dir)
    return model_dir

def make_smpl_model(model_pkl_path, device):
    # smplx expects model_path that contains a subfolder named `smpl`.
    # If a direct file path like .../smpl/SMPL_NEUTRAL.pkl is provided,
    # lift one directory up to .../models.
    model_dir = os.path.dirname(model_pkl_path)
    if os.path.basename(model_dir).lower() == "smpl":
        model_dir = os.path.dirname(model_dir)

    model = smplx.create(
        model_path=model_dir,
        model_type="smpl",
        gender="neutral",
        ext="pkl",
        use_pca=False,
        batch_size=1,
    ).to(device)
    model.eval()
    return model


def pose24x3_to_vertices(smpl_model, pose24x3, device):
    # pose24x3: [24, 3] axis-angle (root + 23 body joints)
    pose = torch.tensor(pose24x3, dtype=torch.float32, device=device).view(24, 3)
    global_orient = pose[0:1, :].reshape(1, 3)
    body_pose = pose[1:, :].reshape(1, -1)  # [1, 69]

    with torch.no_grad():
        out = smpl_model(
            global_orient=global_orient,
            body_pose=body_pose,
            betas=torch.zeros((1, 10), dtype=torch.float32, device=device),
            transl=torch.zeros((1, 3), dtype=torch.float32, device=device),
            return_verts=True,
        )
    verts = out.vertices[0].detach().cpu().numpy()
    return verts


def add_mesh(ax, verts, faces, color, alpha=0.9):
    tris = verts[faces]
    mesh = Poly3DCollection(tris, linewidths=0.02, alpha=alpha)
    mesh.set_facecolor(color)
    mesh.set_edgecolor((0, 0, 0, 0.03))
    ax.add_collection3d(mesh)


In [56]:

# -------------------------
# Resolve paths
# -------------------------
DATA_PATH = resolve_first_existing(DATA_PATH_CANDIDATES)
MODEL_PATH = resolve_latest_wandb_model("wandb/latest-run/files")
SMPL_MODEL_PATH = resolve_first_existing(SMPL_MODEL_CANDIDATES)

print("DATA_PATH:", DATA_PATH)
print("MODEL_PATH (latest-run):", MODEL_PATH)
print("SMPL_MODEL_PATH:", SMPL_MODEL_PATH)


DATA_PATH: data/LIPD/LIPD_SEQUENCES_256p.pkl
MODEL_PATH (latest-run): wandb/latest-run/files/model_145.pth
SMPL_MODEL_PATH: /home/kdh/despite/smpl/SMPL_NEUTRAL.pkl


In [57]:

# -------------------------
# Load data and model
# -------------------------
with open(DATA_PATH, "rb") as f:
    sequence_datasets = pickle.load(f)
print("Available splits:", list(sequence_datasets.keys()))

model_type_to_modalities = {"S": "SKELETON", "I": "IMU", "P": "PC"}
modalities = [model_type_to_modalities[m].lower() for m in MODEL_TYPE if m not in ["E", "T"]]
print("Modalities:", modalities)

if "skeleton" in modalities:
    if SKELETON_SOURCE == "gt_pose":
        skeleton = model_loader.load_smpl_pose_encoder(EMBED_DIM, NUM_JOINTS, device=device, backbone="transformer")
    else:
        skeleton = model_loader.load_skeleton_encoder(EMBED_DIM, NUM_JOINTS, N_FEATS, device=device)
else:
    skeleton = None

imu = model_loader.load_imu_encoder(EMBED_DIM, device=device) if "imu" in modalities else None
pc = model_loader.load_pst_transformer(EMBED_DIM, device=device) if "pc" in modalities else None
model = SPITE.instantiate_binder(modalities, False, imu, pc, skeleton, None).to(device)

ckpt = torch.load(MODEL_PATH, map_location="cpu")
if isinstance(ckpt, dict):
    for key in ("state_dict", "model_state_dict", "model"):
        if key in ckpt and isinstance(ckpt[key], dict):
            ckpt = ckpt[key]
            break
load_msg = model.load_state_dict(ckpt, strict=False)
print("Checkpoint loaded with strict=False")
print("Missing keys:", len(load_msg.missing_keys), "Unexpected keys:", len(load_msg.unexpected_keys))
missing_skel = [k for k in load_msg.missing_keys if k.startswith("skeleton_encoder")]
missing_imu = [k for k in load_msg.missing_keys if k.startswith("imu_encoder")]
print("Missing skeleton keys:", len(missing_skel), "Missing imu keys:", len(missing_imu))
model.eval()




Available splits: ['ACCAD', 'BMLmovi', 'LIPD_train', 'AIST', 'CMU', 'eLIPD', 'eTC', 'eDIP']
Modalities: ['skeleton', 'imu']
Loading class <class 'src.models.SPITE.SIE_BINDER'>
Checkpoint loaded with strict=False
Missing keys: 0 Unexpected keys: 65
Missing skeleton keys: 0 Missing imu keys: 0


/home/kdh/despite/.venv/lib/python3.8/site-packages/torch/nn/modules/transformer.py:286: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


SIE_BINDER(
  (imu_encoder): IMUEncoder(
    (lstm): LSTM(48, 512, num_layers=2, batch_first=True)
  )
  (skeleton_encoder): SMPLPoseEncoder(
    (encoder): Encoder_TRANSFORMER(
      (skelEmbedding): Linear(in_features=144, out_features=512, bias=True)
      (sequence_pos_encoder): PositionalEncoding(
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (seqTransEncoder): TransformerEncoder(
        (layers): ModuleList(
          (0-7): 8 x TransformerEncoderLayer(
            (self_attn): MultiheadAttention(
              (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
            )
            (linear1): Linear(in_features=512, out_features=1024, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
            (linear2): Linear(in_features=1024, out_features=512, bias=True)
            (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
            (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True

In [58]:

# -------------------------
# Encode windows with gt_pose
# -------------------------
encoded_dataset = matching.encode_all(
    copy.deepcopy(sequence_datasets[DATASET_SPLIT]),
    model,
    window_length=24,
    model_type=MODEL_TYPE,
    skeleton_source=SKELETON_SOURCE,
)

print("Encoded subjects:", len(encoded_dataset))
print("Encoded sequences:", sum(len(v) for v in encoded_dataset.values()))


Encoded subjects: 3
Encoded sequences: 7


In [59]:

# -------------------------
# Build multi-person scene (IMU + SMPL pose windows)
# -------------------------
def sample_scene_imu_smpl(dataset, n_subjects=8, window_size=160, seed=42):
    rng = np.random.default_rng(seed)

    candidates = []
    for subject, seqs in dataset.items():
        for seq, seq_dict in seqs.items():
            if "IMU_EMBS" not in seq_dict or "SKELETON_EMBS" not in seq_dict:
                continue
            if len(seq_dict["IMU_EMBS"]) < window_size or len(seq_dict["SKELETON_EMBS"]) < window_size:
                continue
            candidates.append((subject, seq))

    if not candidates:
        raise ValueError("No candidate sequences with enough length.")

    chosen = rng.choice(len(candidates), size=n_subjects, replace=(len(candidates) < n_subjects))

    scene = {"IMU": [], "SMPL_POSE": [], "IMU_EMBS": [], "SMPL_EMBS": [], "META": []}

    for idx in chosen:
        subject, seq = candidates[int(idx)]
        seq_dict = dataset[subject][seq]
        max_start = len(seq_dict["IMU_EMBS"]) - window_size
        start = int(rng.integers(0, max_start + 1))
        end = start + window_size

        scene["IMU"].append(seq_dict["IMU_W"][start:end])
        scene["SMPL_POSE"].append(seq_dict["SKELETON_W"][start:end])
        scene["IMU_EMBS"].append(seq_dict["IMU_EMBS"][start:end])
        scene["SMPL_EMBS"].append(seq_dict["SKELETON_EMBS"][start:end])
        scene["META"].append((subject, seq, start, end))

    scene["IMU"] = torch.stack(scene["IMU"])
    scene["SMPL_POSE"] = torch.stack(scene["SMPL_POSE"])
    scene["IMU_EMBS"] = torch.stack(scene["IMU_EMBS"])
    scene["SMPL_EMBS"] = torch.stack(scene["SMPL_EMBS"])
    return scene


scene = sample_scene_imu_smpl(encoded_dataset, n_subjects=N_SUBJECTS, window_size=WINDOW_SIZE, seed=SCENE_SEED)

print("IMU:", tuple(scene["IMU"].shape))
print("SMPL_POSE:", tuple(scene["SMPL_POSE"].shape), "(axis-angle)")
print("IMU_EMBS:", tuple(scene["IMU_EMBS"].shape))
print("SMPL_EMBS:", tuple(scene["SMPL_EMBS"].shape))
print("Example meta:", scene["META"][0])


IMU: (8, 160, 24, 48)
SMPL_POSE: (8, 160, 24, 24, 3) (axis-angle)
IMU_EMBS: (8, 160, 512)
SMPL_EMBS: (8, 160, 512)
Example meta: ('921', 'seq2', 245, 405)


In [60]:

# -------------------------
# Frame-wise IMU -> SMPL similarity
# sims_f: [T, N_query, N_candidate]
# -------------------------
imu_embs = torch.nn.functional.normalize(scene["IMU_EMBS"].float(), dim=-1)
smpl_embs = torch.nn.functional.normalize(scene["SMPL_EMBS"].float(), dim=-1)

sims_f = torch.einsum("qtd,std->tqs", imu_embs, smpl_embs).cpu().numpy()
pred_idx = sims_f.argmax(axis=2)

print("sims_f:", sims_f.shape)
print("pred_idx:", pred_idx.shape)


sims_f: (160, 8, 8)
pred_idx: (160, 8)


In [61]:
import sys
import numpy as np
import numpy.core as np_core
import numpy.core.multiarray as np_ma

# chumpy (old) compatibility on numpy>=1.24
for name, val in {
    "bool": bool,
    "int": int,
    "float": float,
    "complex": complex,
    "object": object,
    "str": str,
    "unicode": str,
}.items():
    if not hasattr(np, name):
        setattr(np, name, val)

# pickle compatibility for numpy._core path
sys.modules.setdefault("numpy._core", np_core)
sys.modules.setdefault("numpy._core.multiarray", np_ma)


<module 'numpy.core.multiarray' from '/home/kdh/despite/.venv/lib/python3.8/site-packages/numpy/core/multiarray.py'>

In [62]:

# -------------------------
# Prepare mmhuman3d inputs for scene
# -------------------------
N = scene["SMPL_POSE"].shape[0]
T = scene["SMPL_POSE"].shape[1]

# scene["SMPL_POSE"]: [N, T, 24, 3] (axis-angle), take first frame from each 24-frame window
poses_nt72 = scene["SMPL_POSE"][:, :, 0].cpu().numpy().reshape(N, T, 72)
poses_tn72 = np.transpose(poses_nt72, (1, 0, 2))  # [T, N, 72]

x_offsets = (np.arange(N, dtype=np.float32) - (N - 1) / 2.0) * MMHUMAN_SUBJECT_SPACING
transl_tn3 = np.zeros((T, N, 3), dtype=np.float32)
transl_tn3[:, :, 0] = x_offsets[None, :]
transl_tn3[:, :, 2] = 1.8

MMHUMAN3D_MODEL_DIR = os.path.dirname(SMPL_MODEL_PATH)
if os.path.basename(MMHUMAN3D_MODEL_DIR).lower() != "smpl":
    mmhuman_candidate = os.path.join(MMHUMAN3D_MODEL_DIR, "smpl")
    if os.path.isdir(mmhuman_candidate):
        MMHUMAN3D_MODEL_DIR = mmhuman_candidate
MMHUMAN3D_BODY_CFG = dict(type="smpl", model_path=MMHUMAN3D_MODEL_DIR, num_betas=10)

# Build body model once and reuse it to avoid repeated SMPL initialization warnings.
MMHUMAN3D_BODY_MODEL = None
if MMHUMAN3D_AVAILABLE and build_body_model is not None:
    MMHUMAN3D_BODY_MODEL = build_body_model(MMHUMAN3D_BODY_CFG)

print("poses_tn72:", poses_tn72.shape)
print("transl_tn3:", transl_tn3.shape)
print("MMHUMAN3D_MODEL_DIR:", MMHUMAN3D_MODEL_DIR)


poses_tn72: (160, 8, 72)
transl_tn3: (160, 8, 3)
MMHUMAN3D_MODEL_DIR: /home/kdh/despite/smpl


In [63]:
import warnings
warnings.filterwarnings("ignore", message=".*only 10 shape coefficients.*")

In [64]:

# -------------------------
# Render query videos (mmhuman3d + matching colors)
# -------------------------
if (not MMHUMAN3D_AVAILABLE) or (visualize_smpl_calibration is None) or (MMHUMAN3D_BODY_MODEL is None):
    raise ImportError(
        "mmhuman3d with visualize_smpl_calibration is not available. Install/upgrade it, then rerun this cell.\n"
        "Example: pip install -U mmhuman3d"
    )

imus = scene["IMU"][:, :, 0].cpu().numpy()  # [N, T, 48]

def hex_to_rgb255(hex_color):
    hex_color = hex_color.lstrip("#")
    return np.array([int(hex_color[i:i+2], 16) for i in (0, 2, 4)], dtype=np.uint8)

base_other = np.array([170, 170, 170], dtype=np.uint8)
base_match = hex_to_rgb255(MESH_COLOR_MATCH)
base_query = hex_to_rgb255(MESH_COLOR_QUERY)
base_same = np.array([123, 50, 148], dtype=np.uint8)  # #7b3294

render_resolution = RENDER_RESOLUTION
render_h, render_w = render_resolution

# Fixed frontal OpenCV camera for mmhuman3d calibration render.
FRONTAL_FOCAL_SCALE = 1.35
FRONTAL_CAM_TX = 0.0
FRONTAL_CAM_TY = 0.10
FRONTAL_CAM_TZ = 5.5
CAMERA_VIEW_MODE = "front"  # "front" or "visible"

focal = float(max(render_h, render_w) * FRONTAL_FOCAL_SCALE)
K = np.array([[[focal, 0.0, render_w * 0.5],
               [0.0, focal, render_h * 0.5],
               [0.0, 0.0, 1.0]]], dtype=np.float32)
R = np.eye(3, dtype=np.float32)[None, ...]
T_cam = np.array([[FRONTAL_CAM_TX, FRONTAL_CAM_TY, FRONTAL_CAM_TZ]], dtype=np.float32)

# SMPL shape can be omitted, but explicit zeros keep behavior deterministic.
betas_tn10 = np.zeros((T, N, 10), dtype=np.float32)

def _to_u8_rgba_or_rgb(img):
    img = np.asarray(img).astype(np.float32)
    vmax = float(np.nanmax(img)) if img.size > 0 else 1.0
    if vmax <= 1.5:
        img = np.clip(img, 0.0, 1.0) * 255.0
    else:
        img = np.clip(img, 0.0, 255.0)
    return img

def _white_composite(img):
    # img: HxWx3 or HxWx4 in [0,255] float
    if img.shape[-1] == 4:
        alpha = np.clip(img[..., 3:4] / 255.0, 0.0, 1.0)
        rgb = img[..., :3]
        bg = np.full_like(rgb, 255.0)
        out = rgb * alpha + bg * (1.0 - alpha)
    else:
        out = img[..., :3]
    return np.clip(out, 0.0, 255.0).astype(np.uint8)

def _view_score(rgb_u8, mode="front"):
    mask = np.any(np.abs(rgb_u8.astype(np.int16) - 255) > 6, axis=-1)
    vis = float(mask.mean())
    if vis <= 1e-6:
        return -1e9

    h, w = mask.shape
    ys, xs = np.where(mask)
    y0, y1 = ys.min(), ys.max()
    x0, x1 = xs.min(), xs.max()
    box_h = float(y1 - y0 + 1) / float(h)
    box_w = float(x1 - x0 + 1) / float(w)
    box_area = box_h * box_w
    edge = np.concatenate([mask[0, :], mask[-1, :], mask[:, 0], mask[:, -1]]).mean()

    if mode == "front":
        # Prefer non-empty, not too close (huge), and not touching borders.
        target_area = 0.22
        area_term = max(0.0, 1.0 - abs(box_area - target_area) / target_area)
        score = 2.0 * vis + 1.5 * area_term - 2.5 * float(edge)
    else:
        score = vis - 2.5 * float(edge)
    return float(score)

def _render_probe(R_probe, T_probe):
    rendered_probe = visualize_smpl_calibration(
        poses=poses_tn72[0:1],
        betas=betas_tn10[0:1],
        transl=transl_tn3[0:1],
        body_model=MMHUMAN3D_BODY_MODEL,
        K=K,
        R=R_probe,
        T=T_probe,
        palette=np.tile(base_other[None, :], (N, 1)),
        render_choice="mq",
        resolution=render_resolution,
        output_path=None,
        return_tensor=True,
        no_grad=True,
        batch_size=1,
        device=device,
    )
    if torch.is_tensor(rendered_probe):
        img = rendered_probe[0].detach().cpu().numpy()
    elif isinstance(rendered_probe, np.ndarray):
        img = rendered_probe[0] if rendered_probe.ndim == 4 else rendered_probe
    else:
        img = np.asarray(rendered_probe[0] if isinstance(rendered_probe, (list, tuple)) else rendered_probe)
    img = _to_u8_rgba_or_rgb(img)
    rgb = _white_composite(img)
    return rgb, _view_score(rgb, CAMERA_VIEW_MODE)

# Camera auto-selection to avoid blank render and top-view bias.
R_candidates = [
    np.eye(3, dtype=np.float32)[None, ...],
    np.array([[[1, 0, 0], [0, -1, 0], [0, 0, -1]]], dtype=np.float32),
    np.array([[[-1, 0, 0], [0, 1, 0], [0, 0, -1]]], dtype=np.float32),
    np.array([[[-1, 0, 0], [0, -1, 0], [0, 0, 1]]], dtype=np.float32),
    np.array([[[1, 0, 0], [0, 0, -1], [0, 1, 0]]], dtype=np.float32),
    np.array([[[1, 0, 0], [0, 0, 1], [0, -1, 0]]], dtype=np.float32),
]
T_candidates = []
for tz in [3.0, 4.5, 6.0, 8.0, 10.0, -3.0, -4.5, -6.0, -8.0, -10.0]:
    for ty in [0.0, 0.2, -0.2, 0.8, -0.8]:
        T_candidates.append(np.array([[FRONTAL_CAM_TX, ty, tz]], dtype=np.float32))

best_score = -1e9
best_R = R
best_T = T_cam
for R_try in R_candidates:
    for T_try in T_candidates:
        try:
            _, sc = _render_probe(R_try, T_try)
            if sc > best_score:
                best_score = sc
                best_R = R_try
                best_T = T_try
        except Exception:
            continue

R = best_R
T_cam = best_T
print(f"[camera-auto] mode={CAMERA_VIEW_MODE}, score={best_score:.5f}, T={T_cam.reshape(-1).tolist()}")

for query_idx in range(N):
    output_video = f"{OUTPUT_PREFIX}_mmhuman3d_query_{query_idx}.mp4"
    frames = []

    imu_seq = imus[query_idx]
    imu_plot_ch = list(range(36, 48))

    for t in tqdm(range(T), desc=f"Render Query {query_idx}"):
        query_sims = sims_f[t, query_idx]
        matched_idx = int(pred_idx[t, query_idx])

        palette = np.tile(base_other[None, :], (N, 1))
        palette[matched_idx] = base_match
        if query_idx == matched_idx:
            palette[query_idx] = base_same
        else:
            palette[query_idx] = base_query

        rendered = visualize_smpl_calibration(
            poses=poses_tn72[t:t+1],
            betas=betas_tn10[t:t+1],
            transl=transl_tn3[t:t+1],
            body_model=MMHUMAN3D_BODY_MODEL,
            K=K,
            R=R,
            T=T_cam,
            palette=palette,
            render_choice="mq",
            resolution=render_resolution,
            output_path=None,
            return_tensor=True,
            no_grad=True,
            batch_size=1,
            device=device,
        )

        if torch.is_tensor(rendered):
            top_img = rendered[0].detach().cpu().numpy()
        elif isinstance(rendered, np.ndarray):
            top_img = rendered[0] if rendered.ndim == 4 else rendered
        else:
            top_img = np.asarray(rendered[0] if isinstance(rendered, (list, tuple)) else rendered)
        if top_img.shape[-1] == 4:
            # RGBA -> RGB over black background to avoid renderer image-overlay path.
            alpha = top_img[..., 3:4]
            if alpha.dtype != np.float32 and alpha.dtype != np.float64:
                alpha = alpha.astype(np.float32)
            if float(np.nanmax(alpha)) > 1.0:
                alpha = np.clip(alpha / 255.0, 0.0, 1.0)
            rgb = top_img[..., :3].astype(np.float32)
            top_img = (rgb * alpha).astype(np.uint8)
        if top_img.dtype != np.uint8:
            # mmhuman3d may return float in [0,1] or [0,255].
            vmax = float(np.nanmax(top_img)) if top_img.size > 0 else 1.0
            if vmax <= 1.5:
                top_img = (np.clip(top_img, 0.0, 1.0) * 255.0).astype(np.uint8)
            else:
                top_img = np.clip(top_img, 0.0, 255.0).astype(np.uint8)

        fig = plt.figure(figsize=(12, 8))
        gs = GridSpec(2, 2, height_ratios=[2, 1], hspace=0.28)

        # --- Top: mmhuman3d SMPL render ---
        ax_top = fig.add_subplot(gs[0, :])
        ax_top.imshow(top_img)
        ax_top.axis("off")
        ax_top.set_title(f"IMU->SMPL Mesh Matching | Query S{query_idx+1} | Pred S{matched_idx+1}", fontsize=13, pad=10)

        # --- Bottom-left: IMU ---
        ax_imu = fig.add_subplot(gs[1, 0])
        for c_idx in imu_plot_ch:
            ax_imu.plot(imu_seq[:t+1, c_idx], linewidth=1.2)
        ax_imu.set_xlim(0, T)
        ax_imu.set_ylim(float(np.min(imu_seq)), float(np.max(imu_seq)))
        ax_imu.set_title(f"Query IMU (S{query_idx+1})", fontsize=12, fontweight="bold")
        ax_imu.tick_params(labelsize=9)

        # --- Bottom-right: similarity bar ---
        ax_bar = fig.add_subplot(gs[1, 1])
        bar_colors = [MESH_COLOR_OTHER] * N
        bar_colors[matched_idx] = MESH_COLOR_MATCH
        if query_idx != matched_idx:
            bar_colors[query_idx] = MESH_COLOR_QUERY
        else:
            bar_colors[query_idx] = "#7b3294"

        bars = ax_bar.bar(np.arange(N), query_sims, color=bar_colors)
        bars[matched_idx].set_edgecolor("black")
        bars[matched_idx].set_linewidth(2)

        ax_bar.set_ylim(0.0, 1.0)
        ax_bar.set_xticks(np.arange(N))
        ax_bar.set_xticklabels([f"S{i+1}" for i in range(N)])
        ax_bar.set_title("Cosine Similarity to Query IMU", fontsize=12, fontweight="bold")
        ax_bar.tick_params(labelsize=9)

        fig.tight_layout()
        fig.canvas.draw()
        frame = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
        frame = frame.reshape(fig.canvas.get_width_height()[::-1] + (3,))
        frames.append(frame)
        plt.close(fig)

    imageio.mimsave(output_video, frames, fps=FPS)
    print("Saved:", output_video)


KeyboardInterrupt: 


## Notes

- `SKELETON_SOURCE = "gt_pose"`로 설정되어 있어서, 매칭/시각화 모두 `gt_pose` 기준입니다.
- 체크포인트는 `wandb/latest-run/files/model_*.pth` 중 가장 큰 epoch를 자동 선택합니다.
- 렌더가 느리면 `MESH_FACE_STRIDE`를 더 크게 (`12`, `16`) 바꾸세요.


In [65]:

# -------------------------
# Single-frame preview helper
# -------------------------
def preview_single_frame(query_idx=0, frame_idx=0, view_mode="front", use_auto_camera=True):
    if (not MMHUMAN3D_AVAILABLE) or (visualize_smpl_calibration is None) or (MMHUMAN3D_BODY_MODEL is None):
        raise ImportError(
            "mmhuman3d with visualize_smpl_calibration is not available. Install/upgrade it first."
        )

    render_h, render_w = RENDER_RESOLUTION
    focal = float(max(render_h, render_w) * FRONTAL_FOCAL_SCALE)
    K_dbg = np.array([[[focal, 0.0, render_w * 0.5],
                      [0.0, focal, render_h * 0.5],
                      [0.0, 0.0, 1.0]]], dtype=np.float32)
    R_dbg = np.eye(3, dtype=np.float32)[None, ...]
    T_dbg = np.array([[FRONTAL_CAM_TX, FRONTAL_CAM_TY, FRONTAL_CAM_TZ]], dtype=np.float32)

    if 'betas_tn10' not in globals():
        _T, _N = poses_tn72.shape[:2]
        _betas = np.zeros((_T, _N, 10), dtype=np.float32)
    else:
        _betas = betas_tn10

    base_other = np.array([170, 170, 170], dtype=np.uint8)
    base_match = np.array([int(MESH_COLOR_MATCH.lstrip('#')[i:i+2], 16) for i in (0, 2, 4)], dtype=np.uint8)
    base_query = np.array([int(MESH_COLOR_QUERY.lstrip('#')[i:i+2], 16) for i in (0, 2, 4)], dtype=np.uint8)
    base_same = np.array([123, 50, 148], dtype=np.uint8)

    query_sims = sims_f[frame_idx, query_idx]
    matched_idx = int(pred_idx[frame_idx, query_idx])
    N_local = poses_tn72.shape[1]

    palette = np.tile(base_other[None, :], (N_local, 1))
    palette[matched_idx] = base_match
    palette[query_idx] = base_same if query_idx == matched_idx else base_query

    def _score_render(img_arr, mode="front"):
        img_arr = img_arr.astype(np.float32)
        vmax = float(np.nanmax(img_arr)) if img_arr.size > 0 else 1.0
        if vmax <= 1.5:
            img_arr = np.clip(img_arr, 0.0, 1.0) * 255.0
        else:
            img_arr = np.clip(img_arr, 0.0, 255.0)
        if img_arr.shape[-1] == 4:
            a = np.clip(img_arr[..., 3:4] / 255.0, 0.0, 1.0)
            img_arr = img_arr[..., :3] * a + 255.0 * (1.0 - a)
        rgb = np.clip(img_arr[..., :3], 0.0, 255.0).astype(np.uint8)

        mask = np.any(np.abs(rgb.astype(np.int16) - 255) > 6, axis=-1)
        vis = float(mask.mean())
        if vis <= 1e-6:
            return rgb, -1e9

        h, w = mask.shape
        ys, xs = np.where(mask)
        y0, y1 = ys.min(), ys.max()
        x0, x1 = xs.min(), xs.max()
        box_h = float(y1 - y0 + 1) / float(h)
        box_w = float(x1 - x0 + 1) / float(w)
        box_area = box_h * box_w
        edge = np.concatenate([mask[0, :], mask[-1, :], mask[:, 0], mask[:, -1]]).mean()

        if mode == "front":
            target_area = 0.22
            area_term = max(0.0, 1.0 - abs(box_area - target_area) / target_area)
            sc = 2.0 * vis + 1.5 * area_term - 2.5 * float(edge)
        else:
            sc = vis - 2.5 * float(edge)
        return rgb, float(sc)

    best_sc = -1e9
    best_R, best_T = R_dbg, T_dbg
    best_img = None

    if use_auto_camera:
        R_candidates = [
            np.eye(3, dtype=np.float32)[None, ...],
            np.array([[[1, 0, 0], [0, -1, 0], [0, 0, -1]]], dtype=np.float32),
            np.array([[[-1, 0, 0], [0, 1, 0], [0, 0, -1]]], dtype=np.float32),
            np.array([[[-1, 0, 0], [0, -1, 0], [0, 0, 1]]], dtype=np.float32),
            np.array([[[1, 0, 0], [0, 0, -1], [0, 1, 0]]], dtype=np.float32),
            np.array([[[1, 0, 0], [0, 0, 1], [0, -1, 0]]], dtype=np.float32),
        ]
        T_candidates = []
        for tz in [3.0, 4.5, 6.0, 8.0, 10.0, -3.0, -4.5, -6.0, -8.0, -10.0]:
            for ty in [0.0, 0.2, -0.2, 0.8, -0.8]:
                T_candidates.append(np.array([[FRONTAL_CAM_TX, ty, tz]], dtype=np.float32))

        for R_try in R_candidates:
            for T_try in T_candidates:
                try:
                    _r = visualize_smpl_calibration(
                        poses=poses_tn72[frame_idx:frame_idx+1],
                        betas=_betas[frame_idx:frame_idx+1],
                        transl=transl_tn3[frame_idx:frame_idx+1],
                        body_model=MMHUMAN3D_BODY_MODEL,
                        K=K_dbg,
                        R=R_try,
                        T=T_try,
                        palette=palette,
                        render_choice="mq",
                        resolution=RENDER_RESOLUTION,
                        output_path=None,
                        return_tensor=True,
                        no_grad=True,
                        batch_size=1,
                        device=device,
                    )
                    if torch.is_tensor(_r):
                        _img = _r[0].detach().cpu().numpy()
                    elif isinstance(_r, np.ndarray):
                        _img = _r[0] if _r.ndim == 4 else _r
                    else:
                        _img = np.asarray(_r[0] if isinstance(_r, (list, tuple)) else _r)
                    _rgb, _sc = _score_render(_img, mode=view_mode)
                    if _sc > best_sc:
                        best_sc = _sc
                        best_R, best_T = R_try, T_try
                        best_img = _rgb
                except Exception:
                    continue

    print(f"[preview camera-auto] mode={view_mode}, score={best_sc:.5f}, T={best_T.reshape(-1).tolist()}")

    rendered = visualize_smpl_calibration(
        poses=poses_tn72[frame_idx:frame_idx+1],
        betas=_betas[frame_idx:frame_idx+1],
        transl=transl_tn3[frame_idx:frame_idx+1],
        body_model=MMHUMAN3D_BODY_MODEL,
        K=K_dbg,
        R=best_R,
        T=best_T,
        palette=palette,
        render_choice="mq",
        resolution=RENDER_RESOLUTION,
        output_path=None,
        return_tensor=True,
        no_grad=True,
        batch_size=1,
        device=device,
    )

    if torch.is_tensor(rendered):
        top_img = rendered[0].detach().cpu().numpy()
    elif isinstance(rendered, np.ndarray):
        top_img = rendered[0] if rendered.ndim == 4 else rendered
    else:
        top_img = np.asarray(rendered[0] if isinstance(rendered, (list, tuple)) else rendered)

    top_img = top_img.astype(np.float32)
    vmax = float(np.nanmax(top_img)) if top_img.size > 0 else 1.0
    if vmax <= 1.5:
        top_img = np.clip(top_img, 0.0, 1.0) * 255.0
    else:
        top_img = np.clip(top_img, 0.0, 255.0)

    if top_img.shape[-1] == 4:
        alpha = np.clip(top_img[..., 3:4] / 255.0, 0.0, 1.0)
        rgb = top_img[..., :3]
        bg = np.full_like(rgb, 255.0)
        top_img = rgb * alpha + bg * (1.0 - alpha)

    top_img = np.clip(top_img, 0.0, 255.0).astype(np.uint8)
    if best_img is not None and np.any(np.abs(top_img.astype(np.int16) - 255) > 6) == 0:
        top_img = best_img

    plt.figure(figsize=(10, 5))
    plt.imshow(top_img)
    plt.axis('off')
    plt.title(f'Single Frame Preview | Query S{query_idx+1} | Frame {frame_idx} | Pred S{matched_idx+1}')
    plt.show()

    print('query_idx:', query_idx, 'frame_idx:', frame_idx, 'matched_idx:', matched_idx)
    print('similarities:', np.round(query_sims, 3))

# example
preview_single_frame(query_idx=0, frame_idx=0, view_mode="front", use_auto_camera=True)


KeyboardInterrupt: 

In [67]:
# -------------------------
# Manual camera tuner (Orbit + LookAt)
# -------------------------
def _normalize(v, eps=1e-8):
    n = float(np.linalg.norm(v))
    if n < eps:
        return v * 0.0
    return v / n


def look_at_RT_opencv(eye, target, up_world=np.array([0.0, 1.0, 0.0], dtype=np.float32), roll_deg=0.0):
    """Build OpenCV extrinsics from camera center (eye) and look-at target.

    OpenCV camera axes: +x right, +y down, +z forward.
    World->Camera: X_cam = R * X_world + T, with T = -R * eye.
    """
    eye = np.asarray(eye, dtype=np.float32)
    target = np.asarray(target, dtype=np.float32)
    up_world = _normalize(np.asarray(up_world, dtype=np.float32))

    forward = _normalize(target - eye)  # camera +z in world
    right = _normalize(np.cross(forward, up_world))
    if float(np.linalg.norm(right)) < 1e-6:
        # Degenerate when forward and up are parallel
        right = np.array([1.0, 0.0, 0.0], dtype=np.float32)
    up_cam_world = _normalize(np.cross(right, forward))  # camera up in world

    # Optional roll around forward axis
    if abs(float(roll_deg)) > 1e-6:
        th = np.deg2rad(float(roll_deg))
        c, s = np.cos(th), np.sin(th)
        r2 = c * right - s * up_cam_world
        u2 = s * right + c * up_cam_world
        right, up_cam_world = _normalize(r2), _normalize(u2)

    down = -up_cam_world  # OpenCV +y is downward
    R = np.stack([right, down, forward], axis=0).astype(np.float32)
    T = (-R @ eye.reshape(3, 1)).reshape(3).astype(np.float32)
    return R[None, ...], T[None, ...]


def _render_single_frame_rgb(query_idx, frame_idx, K_mat, R_mat, T_vec):
    if (not MMHUMAN3D_AVAILABLE) or (visualize_smpl_calibration is None) or (MMHUMAN3D_BODY_MODEL is None):
        raise ImportError(
            "mmhuman3d with visualize_smpl_calibration/body_model is not ready."
        )

    if 'betas_tn10' not in globals():
        _T, _N = poses_tn72.shape[:2]
        _betas = np.zeros((_T, _N, 10), dtype=np.float32)
    else:
        _betas = betas_tn10

    base_other = np.array([170, 170, 170], dtype=np.uint8)
    base_match = np.array([int(MESH_COLOR_MATCH.lstrip('#')[i:i+2], 16) for i in (0, 2, 4)], dtype=np.uint8)
    base_query = np.array([int(MESH_COLOR_QUERY.lstrip('#')[i:i+2], 16) for i in (0, 2, 4)], dtype=np.uint8)
    base_same = np.array([123, 50, 148], dtype=np.uint8)

    matched_idx = int(pred_idx[frame_idx, query_idx])
    N_local = poses_tn72.shape[1]

    palette = np.tile(base_other[None, :], (N_local, 1))
    palette[matched_idx] = base_match
    palette[query_idx] = base_same if query_idx == matched_idx else base_query

    rendered = visualize_smpl_calibration(
        poses=poses_tn72[frame_idx:frame_idx+1],
        betas=_betas[frame_idx:frame_idx+1],
        transl=transl_tn3[frame_idx:frame_idx+1],
        body_model=MMHUMAN3D_BODY_MODEL,
        K=K_mat,
        R=R_mat,
        T=T_vec,
        palette=palette,
        render_choice="mq",
        resolution=RENDER_RESOLUTION,
        output_path=None,
        return_tensor=True,
        no_grad=True,
        batch_size=1,
        device=device,
    )

    if torch.is_tensor(rendered):
        img = rendered[0].detach().cpu().numpy()
    elif isinstance(rendered, np.ndarray):
        img = rendered[0] if rendered.ndim == 4 else rendered
    else:
        img = np.asarray(rendered[0] if isinstance(rendered, (list, tuple)) else rendered)

    img = img.astype(np.float32)
    vmax = float(np.nanmax(img)) if img.size > 0 else 1.0
    if vmax <= 1.5:
        img = np.clip(img, 0.0, 1.0) * 255.0
    else:
        img = np.clip(img, 0.0, 255.0)

    if img.shape[-1] == 4:
        alpha = np.clip(img[..., 3:4] / 255.0, 0.0, 1.0)
        rgb = img[..., :3]
        bg = np.full_like(rgb, 255.0)
        img = rgb * alpha + bg * (1.0 - alpha)

    img = np.clip(img, 0.0, 255.0).astype(np.uint8)
    return img, matched_idx


def preview_single_frame_orbit(
    query_idx=0,
    frame_idx=0,
    azim_deg=0.0,
    elev_deg=10.0,
    dist=10.0,
    target_x=None,
    target_y=None,
    target_z=None,
    roll_deg=0.0,
    focal_scale=1.35,
):
    h, w = RENDER_RESOLUTION
    focal = float(max(h, w) * focal_scale)
    K_mat = np.array([[[focal, 0.0, w * 0.5],
                       [0.0, focal, h * 0.5],
                       [0.0, 0.0, 1.0]]], dtype=np.float32)

    # Default target: center of all subjects for this frame
    centers = transl_tn3[frame_idx]  # [N, 3]
    center = np.mean(centers, axis=0).astype(np.float32)
    tx = float(center[0]) if target_x is None else float(target_x)
    ty = float(center[1] + 0.9) if target_y is None else float(target_y)
    tz = float(center[2]) if target_z is None else float(target_z)
    target = np.array([tx, ty, tz], dtype=np.float32)

    az = np.deg2rad(float(azim_deg))
    el = np.deg2rad(float(elev_deg))
    d = float(dist)

    # Spherical orbit around target (Y-up world assumption)
    eye = np.array([
        target[0] + d * np.cos(el) * np.sin(az),
        target[1] + d * np.sin(el),
        target[2] + d * np.cos(el) * np.cos(az),
    ], dtype=np.float32)

    R_mat, T_vec = look_at_RT_opencv(eye=eye, target=target, roll_deg=roll_deg)

    rgb, matched_idx = _render_single_frame_rgb(query_idx, frame_idx, K_mat, R_mat, T_vec)

    plt.figure(figsize=(10, 5))
    plt.imshow(rgb)
    plt.axis('off')
    plt.title(f'Orbit Camera | Query S{query_idx+1} | Frame {frame_idx} | Pred S{matched_idx+1}')
    plt.show()

    print('eye:', eye.tolist())
    print('target:', target.tolist())
    print('K=')
    print(K_mat[0])
    print('R=')
    print(R_mat[0])
    print('T=', T_vec.reshape(-1).tolist())
    return K_mat, R_mat, T_vec, eye, target


def launch_camera_tuner(query_idx=0, frame_idx=0):
    try:
        import ipywidgets as widgets
        from IPython.display import display
    except Exception as e:
        print('ipywidgets가 필요합니다. pip install ipywidgets')
        print('fallback: preview_single_frame_orbit(...)를 직접 호출하세요.')
        print('err:', e)
        return

    center = np.mean(transl_tn3[frame_idx], axis=0).astype(np.float32)

    sliders = {
        'azim_deg': widgets.FloatSlider(description='azim', min=-180, max=180, step=1, value=0.0, continuous_update=False),
        'elev_deg': widgets.FloatSlider(description='elev', min=-80, max=80, step=1, value=10.0, continuous_update=False),
        'dist': widgets.FloatSlider(description='dist', min=2.0, max=30.0, step=0.1, value=10.0, continuous_update=False),
        'target_x': widgets.FloatSlider(description='target_x', min=-10.0, max=10.0, step=0.05, value=float(center[0]), continuous_update=False),
        'target_y': widgets.FloatSlider(description='target_y', min=-5.0, max=8.0, step=0.05, value=float(center[1] + 0.9), continuous_update=False),
        'target_z': widgets.FloatSlider(description='target_z', min=-5.0, max=12.0, step=0.05, value=float(center[2]), continuous_update=False),
        'roll_deg': widgets.FloatSlider(description='roll', min=-180, max=180, step=1, value=0.0, continuous_update=False),
        'focal_scale': widgets.FloatSlider(description='fscale', min=0.5, max=3.0, step=0.01, value=1.35, continuous_update=False),
    }

    out = widgets.Output()

    def _update(azim_deg, elev_deg, dist, target_x, target_y, target_z, roll_deg, focal_scale):
        with out:
            out.clear_output(wait=True)
            preview_single_frame_orbit(
                query_idx=query_idx,
                frame_idx=frame_idx,
                azim_deg=azim_deg,
                elev_deg=elev_deg,
                dist=dist,
                target_x=target_x,
                target_y=target_y,
                target_z=target_z,
                roll_deg=roll_deg,
                focal_scale=focal_scale,
            )

    ui = widgets.VBox(list(sliders.values()))
    widgets.interactive_output(_update, sliders)
    display(ui, out)


# usage
# preview_single_frame_orbit(query_idx=0, frame_idx=0, azim_deg=0, elev_deg=10, dist=10)
launch_camera_tuner(query_idx=0, frame_idx=0)



Output()